In [9]:
# Jupyter cell — VS Code

from pathlib import Path
import re
import pandas as pd

# ---------- Paths ----------
SAMPLE_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10.csv")
OUT_XLSX   = SAMPLE_CSV.with_name("stratified_sample_moe10_output.xlsx")

CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")
TESTS_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Test_Files")

# ---------- Helpers ----------
YAML_EXTS   = {".yml", ".yaml"}
GRADLE_EXTS = {".gradle"}  # .gradle.kts handled via name.endswith

# Allowed categories (now multi-valued)
EXECENV_ALLOWED = [
    "Emulator_ReactiveCircus",
    "Emulator_Malinskiy",
    "Emulator_DIY",
    "Emulator_Other",
    "Emulator_GMD",
    "GMD_intent",
    "Third Party",
    "Real Device",
    "Unknown",
]
INVOC_ALLOWED = [
    "Gradle_GMD",
    "Gradle",                  # connected*/deviceCheck/Spoon/Marathon/inputs
    "Gradle_BaselineProfile",  # generate*/collect*BaselineProfile or connectedBenchmarkAndroidTest
    "ADB",
    "3P CLIs",
    "Unknown",
]

def truth01(val):
    s = pd.Series([val])
    if s.dtype == bool:
        return int(s.iloc[0])
    t = s.astype(str).str.strip().str.lower()
    return int(t.iloc[0] in {"1","true","t","yes","y","on"})

def _norm_one(x: str, allowed: list) -> str:
    if pd.isna(x) or not str(x).strip():
        return ""
    s = str(x).strip()
    low = s.lower()
    aliases = {
        "reactivecircus": "Emulator_ReactiveCircus",
        "malinskiy": "Emulator_Malinskiy",
        "diy": "Emulator_DIY",
        "other": "Emulator_Other",
        "third party": "Third Party",
        "third_party": "Third Party",
        "third-party": "Third Party",
        "real": "Real Device",
        "real device": "Real Device",
        "gmd": "Emulator_GMD",
        "gmd_intent": "GMD_intent",
        "3p": "3P CLIs",
        "3p clis": "3P CLIs",
        "third party clis": "3P CLIs",
        "baselineprofile": "Gradle_BaselineProfile",
    }
    if low in aliases:
        s = aliases[low]
    for a in allowed:
        if s.lower() == a.lower():
            return a
    return s

def norm_multi(x: str, allowed: list) -> list:
    if pd.isna(x) or not str(x).strip():
        return []
    parts = [p.strip() for p in str(x).split(",") if p.strip()]
    out = []
    seen = set()
    for p in parts:
        n = _norm_one(p, allowed)
        if n and n.lower() not in seen:
            out.append(n)
            seen.add(n.lower())
    return out

def safe_read_text(p: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    return ""

def repo_key_from_filename(fname: str) -> str:
    base = Path(fname).name
    key = base.split("__", 1)[0] if "__" in base else Path(base).stem
    return key.strip().lower()

def index_repo_files(root: Path):
    idx = {}
    if root.exists():
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            is_yaml   = p.suffix.lower() in YAML_EXTS
            is_gradle = (p.suffix.lower() in GRADLE_EXTS) or p.name.lower().endswith(".gradle.kts")
            if is_yaml or is_gradle:
                key = repo_key_from_filename(p.name)
                idx.setdefault(key, []).append(p)
    return idx

def build_androidtest_presence_index(tests_dir: Path):
    present = set()
    if tests_dir.exists():
        for p in tests_dir.iterdir():
            if p.is_file():
                present.add(repo_key_from_filename(p.name))
    return present

# ---------- Comment + sanitizer (match detection code) ----------
EXCLUDED_TASK_SEGMENT_RE = re.compile(
    r'(^|\s)(?:-x|--exclude-task)\s+(["\']?)[:\w\.-]*(?:androidtest|baselineprofile)[\w:\.-]*\2\b',
    re.IGNORECASE | re.MULTILINE,
)
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")

def strip_comments_yaml(text: str) -> str:
    if not text:
        return text
    text = re.sub(r'(?m)^\s*#.*$', '', text)
    text = re.sub(r'(?m)\s#.*$', '', text)
    return text

def strip_comments_gradle(text: str) -> str:
    if not text:
        return text
    text = re.sub(r'/\*.*?\*/', '', text, flags=re.S)
    text = re.sub(r'(?m)^\s*//.*$', '', text)
    return text

def pre_sanitize(text: str) -> str:
    t = EXCLUDED_TASK_SEGMENT_RE.sub(lambda m: (m.group(1) or " "), text or "")
    return GHA_EXPR_RE.sub("", t or "")

def any_hit(text: str, patterns: dict) -> bool:
    return any(re.search(p, text, flags=re.IGNORECASE | re.DOTALL | re.MULTILINE) for p in patterns.values())

def find_any(text: str, pattern_list) -> bool:
    return any(re.search(p, text, flags=re.IGNORECASE | re.DOTALL | re.MULTILINE) for p in pattern_list)

def looks_like_real_device_serial(cmd: str) -> bool:
    m = re.search(r"\badb\s+-s\s+([^\s]+)", cmd, flags=re.IGNORECASE)
    if not m:
        return False
    serial = m.group(1).lower()
    if serial.startswith("emulator-") or serial.startswith("localhost:") or serial.startswith("127."):
        return False
    return True

# --- Patterns aligned with detection taxonomy ---
YAML_SIGNAL_PATTERNS = {
    # Exec-env via actions & services
    "reactivecircus_runner": r"uses:\s*reactivecircus/android-emulator-runner@",
    "malinskiy_runner_a":    r"uses:\s*malinskiy/android-emulator-runner@",
    "malinskiy_runner_b":    r"uses:\s*malinskiy/action-android(?:@|/)",
    # CircleCI Android orb (emulator)
    "circleci_orb":          r"(?mi)^\s*android/start-emulator-and-run-tests\s*:|\bsystem-image\s*:\s*system-images;android-\d+;google_apis;",

    # Robust Gradle invocations
    "gradle_connected_check_only":    r"\bgradle[w]?\b[^\n\r]*\bconnectedCheck\b",
    "gradle_connected_androidtest":   r"\bgradle[w]?\b[^\n\r]*\bconnected[A-Za-z0-9:_-]*AndroidTest\b",
    "gradle_connected_generic":       r"\bgradle[w]?\b[^\n\r]*\bconnected(?:androidtest|check)\b",
    "gradle_device_checks":           r"\bgradle[w]?\b[^\n\r]*\b(?:deviceCheck|allDeviceChecks)\b",

    # Managed devices / variant tasks (GMD signals)
    "gradle_managed_androidtest":     r"\bgradle[w]?\b[^\n\r]*\b(?:managedDevice|device)[\w:-]*AndroidTest\b",
    # gated final segment: not assemble/bundle/connected
    "gradle_variant_androidtest":     r"\bgradle[w]?\b[^\n\r]*\b(?:[:\w-]+:)*(?!(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload))(?<!connected)[A-Za-z0-9][\w-]*AndroidTest\b",

    # Baseline Profile & Benchmark
    "baseline_profile":               r"\b(?:generate|collect)[\w-]*BaselineProfile\b",
    "connected_benchmark":            r"\bconnectedBenchmarkAndroidTest\b",

    # ADB instrumentation CLI
    "adb_instrument":                 r"\badb\s+(?:-s\s+\S+\s+)?shell\s+am\s+instrument\b",

    # Third-party CLIs
    "gcloud_ftl":                     r"\bgcloud\b[^\n\r]*\bfirebase\s+test\s+android\s+run\b",
    "flank":                          r"\bflank\b[^\n\r]*\bandroid\b",
    "saucectl":                       r"\bsaucectl\b",
    "appcenter":                      r"\bappcenter\s+test\s+run\s+android\b",
    "maestro_cloud":                  r"\bmaestro\s+cloud\b",
    "emulator_wtf":                   r"\bemulator\.wtf\b|\bew-cli\b",
    "browserstack":                   r"\bbrowserstack\b|\bbstack\b|\bbstack-sdk\b",

    # DIY emulator hints
    "emulator_launch":                r"\bemulator\b[^\n\r]*(?:-avd\s+\S+|@\S+)",
    "avdmanager":                     r"\bavdmanager\b|\bandroid\s+create\s+avd\b",
    "sdk_sysimg":                     r"\bsdkmanager\b[^\n\r]*system-images;android-",

    # Flutter integration tests (kept)
    "flutter_android":                r"\bflutter\s+(?:drive|test)\b[^\n\r]*(integration_test|/integration_test/)",
    "dart_integ":                     r"\bdart\s+test\b[^\n\r]*(integration_test|/integration_test/)",

    # YAML hints of GMD config presence
    "gmd_yaml":                       r"\b(ManagedVirtualDevice|managedDevices|deviceGroups)\b",
}

# Which YAML patterns count as exec-env or invocation
YAML_EXECENV_KEYS = {
    "reactivecircus_runner","malinskiy_runner_a","malinskiy_runner_b",
    "circleci_orb",
    "emulator_launch","avdmanager","sdk_sysimg","gmd_yaml"
}
YAML_INVOC_KEYS = {
    "gradle_connected_check_only","gradle_connected_androidtest","gradle_connected_generic","gradle_device_checks",
    "gradle_managed_androidtest","gradle_variant_androidtest",
    "baseline_profile","connected_benchmark",
    "adb_instrument","gcloud_ftl","flank","saucectl","appcenter","maestro_cloud","emulator_wtf","browserstack"
}

# Build/Gradle signals (build.gradle / *.kts)
BUILD_SIGNAL_PATTERNS = {
    "testInstrumentationRunner": r"\btestInstrumentationRunner\b\s*(=|\s)\s*['\"][^'\"]+['\"]?",
    "androidTestDependency_str": r"\bandroidTest(?:Implementation|Api|CompileOnly|RuntimeOnly)\s*\(?\s*['\"][^'\"]+['\"]",
    "androidTestDependency_alias": r"\bandroidTest(?:Implementation|Api|CompileOnly|RuntimeOnly)\s*\(\s*[a-zA-Z0-9_.:-]+\s*\)",
    "androidx_test": r"['\"][^'\"]*androidx\.test[^'\"]*['\"]",
    "espresso": r"['\"][^'\"]*espresso[^'\"]*['\"]",
    "uiautomator": r"['\"][^'\"]*uiautomator[^'\"]*['\"]",
    "orchestrator": r"['\"][^'\"]*androidx\.test:orchestrator[^'\"]*['\"]",
    "benchmark": r"['\"][^'\"]*androidx\.benchmark[^'\"]*['\"]",
    "useOrchestrator_true": r"\buseOrchestrator\b\s*(=|\s)\s*true\b",
    "connectedAndroidTest": r"\bconnectedAndroidTest\b",
    # GMD config hints
    "gmd_block_hint": r"\btestOptions\s*\{[^}]*managedDevices\b|testOptions\.managedDevices|ManagedVirtualDevice",
    # Baseline profile gradle tasks (project files sometimes include them in docs/scripts)
    "gradle_baseline_task": r"\b(?:generate|collect)[\w-]*BaselineProfile\b",
}

# ---------- Load sample & build indices ----------
df = pd.read_csv(SAMPLE_CSV, low_memory=False)
if "full_name" not in df.columns:
    raise KeyError("Sample must contain a 'full_name' column.")

cfg_index = index_repo_files(CONFIG_DIR)
at_present = build_androidtest_presence_index(TESTS_DIR)

# ---------- Scan ----------
def collect_hits(text: str, patterns: dict):
    hits = {}
    if not text:
        return hits
    for name, pat in patterns.items():
        found = []
        for m in re.finditer(pat, text, flags=re.IGNORECASE | re.DOTALL | re.MULTILINE):
            start = max(0, m.start() - 80)
            end   = min(len(text), m.end() + 80)
            snippet = text[start:end].replace("\r", " ").strip()
            found.append(snippet)
            if len(found) >= 3:
                break
        if found:
            hits[name] = found
    return hits

keys = df["full_name"].astype(str).str.strip().str.lower()
uniq = keys.unique()

scan_result = {}
for k in uniq:
    yaml_found = 0
    build_found = 0
    texts_yaml_active, texts_gradle_active = [], []
    yaml_sources, build_sources = [], []
    yaml_hits_perfile, build_hits_perfile = {}, {}

    for p in cfg_index.get(k, []):
        raw = safe_read_text(p)
        if not raw:
            continue
        is_yaml   = p.suffix.lower() in YAML_EXTS
        is_gradle = (p.suffix.lower() in GRADLE_EXTS) or p.name.lower().endswith(".gradle.kts")

        if is_yaml:
            t = pre_sanitize(strip_comments_yaml(raw))
            texts_yaml_active.append(t)
            hits = collect_hits(t, YAML_SIGNAL_PATTERNS)
            if hits:
                yaml_hits_perfile[p.name] = hits
                if any(h in YAML_EXECENV_KEYS or h in YAML_INVOC_KEYS for h in hits.keys()):
                    yaml_found = 1
                for h in hits.keys():
                    if h in YAML_EXECENV_KEYS or h in YAML_INVOC_KEYS:
                        yaml_sources.append(f"{p.name}:{h}")

        if is_gradle:
            t = strip_comments_gradle(raw)
            texts_gradle_active.append(t)
            hits = collect_hits(t, BUILD_SIGNAL_PATTERNS)
            if hits:
                build_hits_perfile[p.name] = hits
                build_found = 1
                for h in hits.keys():
                    build_sources.append(f"{p.name}:{h}")

    yaml_text   = "\n".join(texts_yaml_active)
    gradle_text = "\n".join(texts_gradle_active)

    # --- Exec env signals ---
    rc  = bool(re.search(YAML_SIGNAL_PATTERNS["reactivecircus_runner"], yaml_text, re.I|re.S|re.M))
    ml  = bool(re.search(YAML_SIGNAL_PATTERNS["malinskiy_runner_a"], yaml_text, re.I|re.S|re.M)) or \
          bool(re.search(YAML_SIGNAL_PATTERNS["malinskiy_runner_b"], yaml_text, re.I|re.S|re.M))
    orb = bool(re.search(YAML_SIGNAL_PATTERNS["circleci_orb"], yaml_text, re.I|re.S|re.M))
    diy = find_any(yaml_text, [YAML_SIGNAL_PATTERNS["emulator_launch"], YAML_SIGNAL_PATTERNS["avdmanager"], YAML_SIGNAL_PATTERNS["sdk_sysimg"]])
    gmd_yaml_intent = bool(re.search(YAML_SIGNAL_PATTERNS["gmd_yaml"], yaml_text, re.I|re.S|re.M))
    gmd_build = bool(re.search(BUILD_SIGNAL_PATTERNS["gmd_block_hint"], gradle_text, re.I|re.S|re.M))

    third_party = find_any(
        yaml_text,
        [
            YAML_SIGNAL_PATTERNS["gcloud_ftl"], YAML_SIGNAL_PATTERNS["flank"],
            YAML_SIGNAL_PATTERNS["saucectl"], YAML_SIGNAL_PATTERNS["appcenter"],
            YAML_SIGNAL_PATTERNS["maestro_cloud"], YAML_SIGNAL_PATTERNS["emulator_wtf"],
            YAML_SIGNAL_PATTERNS["browserstack"]
        ]
    )

    # Real device via adb -s <serial> (non-emulator)
    real_device = False
    for m in re.finditer(r"\badb\s+-s\s+[^\s]+[^\n\r]*", yaml_text, flags=re.I):
        if looks_like_real_device_serial(m.group(0)):
            real_device = True
            break

    # --- Invocation signals ---
    has_connected = find_any(yaml_text, [
        YAML_SIGNAL_PATTERNS["gradle_connected_check_only"],
        YAML_SIGNAL_PATTERNS["gradle_connected_androidtest"],
        YAML_SIGNAL_PATTERNS["gradle_connected_generic"],
    ]) or bool(re.search(BUILD_SIGNAL_PATTERNS["connectedAndroidTest"], gradle_text, re.I|re.S|re.M))

    has_device_checks = bool(re.search(YAML_SIGNAL_PATTERNS["gradle_device_checks"], yaml_text, re.I|re.S|re.M))

    has_gmd_gradle = find_any(yaml_text, [
        YAML_SIGNAL_PATTERNS["gradle_managed_androidtest"],
        YAML_SIGNAL_PATTERNS["gradle_variant_androidtest"],   # gated to avoid assemble/bundle/connected
    ])

    has_baseline = find_any(yaml_text, [
        YAML_SIGNAL_PATTERNS["baseline_profile"],
        YAML_SIGNAL_PATTERNS["connected_benchmark"],
    ]) or bool(re.search(BUILD_SIGNAL_PATTERNS["gradle_baseline_task"], gradle_text, re.I|re.S|re.M))

    inv_adb = bool(re.search(YAML_SIGNAL_PATTERNS["adb_instrument"], yaml_text, re.I|re.S|re.M))
    inv_3p  = third_party

    # Spoon/Marathon -> Gradle (other)
    has_other_gradle = False
    # (You can add explicit Spoon/Marathon YAML patterns if you want to track them separately)

    # --- Build-only instrumentation hints (deps/runner/orchestrator)
    build_it_deps = any_hit(gradle_text, BUILD_SIGNAL_PATTERNS)

    # --- AT presence (file-level)
    at_found = 1 if k in at_present else 0

    # --- Decide ExecEnv (multi) ---
    exec_env_labels = []
    if third_party:
        exec_env_labels.append("Third Party")
    if real_device:
        exec_env_labels.append("Real Device")
    if rc:
        exec_env_labels.append("Emulator_ReactiveCircus")
    if ml:
        exec_env_labels.append("Emulator_Malinskiy")
    if orb:
        exec_env_labels.append("Emulator_Other")  # distinct marker for CircleCI orb; still emulator generic
    if gmd_build:
        exec_env_labels.append("Emulator_GMD")
    if gmd_yaml_intent and "Emulator_GMD" not in exec_env_labels:
        exec_env_labels.append("GMD_intent")
    if diy:
        exec_env_labels.append("Emulator_DIY")
    # If we saw connected/gradle + some emulator hint but nothing specific, add Other
    if (has_connected or has_device_checks or has_gmd_gradle) and not any(e.startswith("Emulator_") for e in exec_env_labels) and not third_party and not real_device:
        exec_env_labels.append("Emulator_Other")

    # de-dupe, keep order
    seen = set(); exec_env_labels = [x for x in exec_env_labels if not (x in seen or seen.add(x))]
    if not exec_env_labels:
        exec_env_labels = ["Unknown"]

    # --- Decide Invocation (multi) ---
    inv_labels = []
    if has_gmd_gradle:
        # Only tag GMD; connected is a separate label
        inv_labels.append("Gradle_GMD")
    if has_connected or has_device_checks or has_other_gradle:
        inv_labels.append("Gradle")
    if has_baseline:
        inv_labels.append("Gradle_BaselineProfile")
    if inv_adb:
        inv_labels.append("ADB")
    if inv_3p:
        inv_labels.append("3P CLIs")

    seen = set(); inv_labels = [x for x in inv_labels if not (x in seen or seen.add(x))]
    if not inv_labels:
        inv_labels = ["Unknown"]

    # --- CI signal (boolean) ---
    ci_signal = int(any([
        inv_labels != ["Unknown"],
        build_it_deps,
        at_found == 1
    ]))

    scan_result[k] = {
        # legacy booleans
        "YAML_Check": int(yaml_found),
        "Build_Check": int(build_found),
        "AT_Check": int(at_found),
        # new fields (multi, comma-separated)
        "ExecEnv_Check": ",".join(exec_env_labels),
        "Invocation_Check": ",".join(inv_labels),
        "CI_Signal_Check": int(ci_signal),
        # detection sources
        "YAML_Detect_Source": "; ".join(sorted(set(yaml_sources))) if yaml_sources else "",
        "Build_Detect_Source": "; ".join(sorted(set(build_sources))) if build_sources else "",
        # Flutter IT
        "Flutter_IT_Check": int(find_any(yaml_text, [YAML_SIGNAL_PATTERNS["flutter_android"], YAML_SIGNAL_PATTERNS["dart_integ"]])),
    }

# ---------- Map back to rows ----------
df["YAML_Check"]          = [scan_result.get(k, {}).get("YAML_Check", 0) for k in keys]
df["Build_Check"]         = [scan_result.get(k, {}).get("Build_Check", 0) for k in keys]
df["AT_Check"]            = [scan_result.get(k, {}).get("AT_Check", 0) for k in keys]
df["ExecEnv_Check"]       = [scan_result.get(k, {}).get("ExecEnv_Check", "Unknown") for k in keys]
df["Invocation_Check"]    = [scan_result.get(k, {}).get("Invocation_Check", "Unknown") for k in keys]
df["CI_Signal_Check"]     = [scan_result.get(k, {}).get("CI_Signal_Check", 0) for k in keys]
df["Flutter_IT_Check"]    = [scan_result.get(k, {}).get("Flutter_IT_Check", 0) for k in keys]
df["YAML_Detect_Source"]  = [scan_result.get(k, {}).get("YAML_Detect_Source", "") for k in keys]
df["Build_Detect_Source"] = [scan_result.get(k, {}).get("Build_Detect_Source", "") for k in keys]

# ---------- Mismatch Note vs any *_pred columns (now set-aware for multi) ----------
def add_mismatch(mismatches, name, pred, check, cat_allowed=None, multi=False):
    if pd.isna(pred):
        return
    if not multi:
        yp = truth01(pred); yc = truth01(check)
        if yp != yc:
            mismatches.append(f"{name} mismatch (pred={yp}, found={yc})")
    else:
        yp_set = set(norm_multi(pred, cat_allowed))
        yc_set = set(norm_multi(check, cat_allowed))
        if yp_set != yc_set:
            mismatches.append(f"{name} mismatch (pred={','.join(sorted(yp_set)) or '∅'}, found={','.join(sorted(yc_set)) or '∅'})")

notes = []
for _, row in df.iterrows():
    mm = []
    # legacy
    if "YAML_pred"  in df.columns: add_mismatch(mm, "YAML",  row["YAML_pred"],  row["YAML_Check"])
    if "Build_pred" in df.columns: add_mismatch(mm, "Build", row["Build_pred"], row["Build_Check"])
    if "AT_pred"    in df.columns: add_mismatch(mm, "AT",    row["AT_pred"],    row["AT_Check"])
    # new (multi)
    if "ExecEnv_pred"    in df.columns: add_mismatch(mm, "ExecEnv",    row["ExecEnv_pred"],    row["ExecEnv_Check"],    EXECENV_ALLOWED, multi=True)
    if "Invocation_pred" in df.columns: add_mismatch(mm, "Invocation", row["Invocation_pred"], row["Invocation_Check"], INVOC_ALLOWED,    multi=True)
    if "CI_Signal_pred"  in df.columns: add_mismatch(mm, "CI_Signal",  row["CI_Signal_pred"],  row["CI_Signal_Check"])
    if "Flutter_IT_pred" in df.columns: add_mismatch(mm, "Flutter_IT", row["Flutter_IT_pred"], row["Flutter_IT_Check"])
    notes.append("; ".join(mm))

df["Note"] = notes

# ---------- Save Excel next to input ----------
new_cols_order = [
    "YAML_Check","Build_Check","AT_Check",
    "CI_Signal_Check","ExecEnv_Check","Invocation_Check","Flutter_IT_Check",
    "YAML_Detect_Source","Build_Detect_Source",
    "Note"
]
for c in new_cols_order:
    if c not in df.columns:
        df[c] = ""

df.to_excel(OUT_XLSX, index=False)
print(f"[DONE] Wrote: {OUT_XLSX}")


[DONE] Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10_output.xlsx


## accuracy check

In [8]:
# %% [markdown]
# Per-stratum accuracy + per-factor PR/F1 + overall (micro/macro) PR/F1
# Input:  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10_output.xlsx
# Output: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\validation_metrics.xlsx

# %%
import pandas as pd
import numpy as np
from pathlib import Path

# ---------- CONFIG ----------
FOLDER = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final")
IN_XLSX = FOLDER / "stratified_sample_moe10 Manula Reivew Final.xlsx"
OUT_XLSX = FOLDER / "validation_metrics.xlsx"

# ---------- Helpers ----------
def to01(s: pd.Series) -> pd.Series:
    """Robust boolean -> {0,1}."""
    if s.dtype == bool:
        return s.astype(int)
    if pd.api.types.is_numeric_dtype(s):
        return (pd.to_numeric(s, errors="coerce").fillna(0) != 0).astype(int)
    t = s.astype(str).str.strip().str.lower()
    truthy = {"1","true","t","yes","y","on"}
    falsy  = {"0","false","f","no","n","off","","none","null","nan"}
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[t.isin(truthy)] = 1
    out[t.isin(falsy)]  = 0
    # fallback numeric
    num = pd.to_numeric(t.str.replace(r"[^0-9\.\-]+","", regex=True), errors="coerce")
    out = out.where(out.notna(), (num.fillna(0) != 0).astype(int))
    return out.astype(int)

def pr_from_counts(tp, fp, fn):
    prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    rec  = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1   = (2*prec*rec)/(prec+rec) if (pd.notna(prec) and pd.notna(rec) and (prec+rec)>0) else np.nan
    return prec, rec, f1

def counts_for(pred, true):
    tp = int(((pred==1) & (true==1)).sum())
    fp = int(((pred==1) & (true==0)).sum())
    fn = int(((pred==0) & (true==1)).sum())
    tn = int(((pred==0) & (true==0)).sum())
    return tp, fp, fn, tn

# ---------- Load ----------
df = pd.read_excel(IN_XLSX)

need = ["YAML_pred","Build_pred","AT_pred","YAML_Check","Build_Check","AT_Check"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"Missing column(s) in input: {missing}")

# Normalize to 0/1
Yp = to01(df["YAML_pred"]);  Bp = to01(df["Build_pred"]);  Ap = to01(df["AT_pred"])
Yt = to01(df["YAML_Check"]); Bt = to01(df["Build_Check"]); At = to01(df["AT_Check"])

# Ensure stratum label (from predictions if not provided)
if "stratum" in df.columns:
    strata = df["stratum"].astype(str)
else:
    strata = pd.Series([f"Y{y}_B{b}_A{a}" for y,b,a in zip(Yp,Bp,Ap)], index=df.index)

# ---------- 1) Per-stratum accuracy (exact triplet match) ----------
triplet_match = (Yp.eq(Yt) & Bp.eq(Bt) & Ap.eq(At)).astype(int)
per_stratum = (
    pd.DataFrame({"stratum": strata, "match": triplet_match})
      .groupby("stratum", dropna=False)
      .agg(n=("match","size"), matches=("match","sum"))
      .reset_index()
      .sort_values("stratum", ignore_index=True)
)
per_stratum["accuracy"] = per_stratum["matches"] / per_stratum["n"]
overall_triplet_accuracy = float(triplet_match.mean())

# ---------- 2) Per-factor precision/recall/F1 (whole sample) ----------
rows = []
for label, pred, true in [("YAML",Yp,Yt), ("Build",Bp,Bt), ("AT",Ap,At)]:
    tp, fp, fn, tn = counts_for(pred, true)
    prec, rec, f1 = pr_from_counts(tp, fp, fn)
    rows.append({
        "factor": label,
        "N": int(len(pred)),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(prec,4) if pd.notna(prec) else np.nan,
        "recall":    round(rec, 4) if pd.notna(rec) else np.nan,
        "f1":        round(f1,   4) if pd.notna(f1) else np.nan,
    })
per_factor_metrics = pd.DataFrame(rows)

# ---------- 3) OVERALL precision/recall/F1 (whole sample) ----------
# Micro-average (pool all Y,B,AT decisions):
tp = per_factor_metrics["TP"].sum()
fp = per_factor_metrics["FP"].sum()
fn = per_factor_metrics["FN"].sum()
prec_micro, rec_micro, f1_micro = pr_from_counts(tp, fp, fn)

# Macro-average (unweighted mean across factors):
prec_macro = per_factor_metrics["precision"].mean(skipna=True)
rec_macro  = per_factor_metrics["recall"].mean(skipna=True)
f1_macro   = per_factor_metrics["f1"].mean(skipna=True)

overall_metrics = pd.DataFrame([
    {"overall_type":"micro", "precision":round(prec_micro,4), "recall":round(rec_micro,4), "f1":round(f1_micro,4)},
    {"overall_type":"macro", "precision":round(prec_macro,4),  "recall":round(rec_macro,4),  "f1":round(f1_macro,4)},
    {"overall_type":"triplet_exact_accuracy", "precision":np.nan, "recall":np.nan, "f1":np.nan}
])
# Attach the triplet exact-match accuracy as a side note:
overall_note = pd.DataFrame([{"overall_triplet_exact_accuracy": round(overall_triplet_accuracy,4),
                              "rows": len(df)}])

# ---------- Save ----------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    per_stratum.to_excel(xw, sheet_name="Stratum_Accuracy", index=False)
    per_factor_metrics.to_excel(xw, sheet_name="Per_Factor_PRF1", index=False)
    overall_metrics.to_excel(xw, sheet_name="Overall_PRF1", index=False)
    overall_note.to_excel(xw, sheet_name="Overall_Note", index=False)

print(f"[OK] Saved metrics → {OUT_XLSX}")
print("Overall (micro) P/R/F1:", round(prec_micro,4), round(rec_micro,4), round(f1_micro,4))
print("Overall (macro) P/R/F1:", round(prec_macro,4), round(rec_macro,4), round(f1_macro,4))
print("Triplet exact-match accuracy:", round(overall_triplet_accuracy,4))


[OK] Saved metrics → C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\validation_metrics.xlsx
Overall (micro) P/R/F1: 0.989 0.9908 0.9899
Overall (macro) P/R/F1: 0.9906 0.9895 0.99
Triplet exact-match accuracy: 0.9711
